In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub
import os
from tqdm import tqdm

%matplotlib inline

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
csv_path = os.path.join(path, "Q1_data.csv")

df = pd.read_csv(csv_path)



In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 5: Write your code here:
def check_target_distribution(df, target_column):
  df[target_column].hist(bins=30, edgecolor='black')

  plt.title(f"Target Distribution ({target_column})")
  plt.xlabel(target_column)
  plt.ylabel("Frequency")
  plt.grid(False)

  plt.show()

check_target_distribution(df, "Delivery_Time")

In [ ]:
# Task 1: Write your code here:
df = df.drop(['Order_ID'], axis=1)
df.head()

In [ ]:
# Task 2: Write your code here:
def check_missing_values(df):
  missing_values = df.isnull().sum()
  print("Missing Values per Column:")
  print(missing_values[missing_values > 0])
  if missing_values.any():
    print("\nHandle Missing Values as needed.")
  else:
    print("\nNo Missing Values Found.")

check_missing_values(df)


df_clean = df.dropna().copy()

In [ ]:
#Check if it is handeled
def check_missing_values(df):
  missing_values = df.isnull().sum()
  print("Missing Values per Column:")
  print(missing_values[missing_values > 0])
  if missing_values.any():
    print("\nHandle Missing Values as needed.")
  else:
    print("\nNo Missing Values Found.")

check_missing_values(df_clean)

In [ ]:
df_clean

In [ ]:
# Task 3: Write your code here:
# 4. Do we have duplicate samples?
def check_duplicates(df_clean):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df_clean.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df_clean)

In [ ]:
from sklearn.preprocessing import LabelEncoder
categorical_cols = df_clean.select_dtypes(include=["object"]).columns

print("Categorical Columns:", list(categorical_cols))
label_encoders = {}
for col in categorical_cols:
  le = LabelEncoder()
  df_clean[col] = le.fit_transform(df_clean[col])
  label_encoders[col] = le

df_clean

In [ ]:
# Task 4: Write your code here:
categorical_cols = df_clean.select_dtypes(include=["object"]).columns

print("Categorical Columns:", list(categorical_cols))

In [ ]:
# Task 5: Write your code here:
from sklearn.preprocessing import StandardScaler

numerical_cols = df_clean.select_dtypes(include=["int64", "float64"]).columns.drop("Delivery_Time")  # DON'T SCALE THE TARGET

scaler = StandardScaler()
df_clean[numerical_cols] = scaler.fit_transform(df_clean[numerical_cols])
df_clean.head()

In [ ]:
# Task 6: Write your code here:
def check_target_distribution(df_clean, target_column):
  df_clean[target_column].hist(bins=30, edgecolor='black')

  plt.title(f"Target Distribution ({target_column})")
  plt.xlabel(target_column)
  plt.ylabel("Frequency")
  plt.grid(False)

  plt.show()

check_target_distribution(df_clean, "Delivery_Time")

In [ ]:
# Task 1: Write your code here:
X = df_clean.drop("Delivery_Time", axis=1).astype(float)
y = df_clean['Delivery_Time'].astype(float)

In [ ]:
# Task 2,3,4,5: Write your code here:
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error as sklearn_mse, mean_absolute_error, r2_score
lr_mae = []
models = {

  "Random Forest Regressor": RandomForestRegressor(n_estimators=200)

}
all_results = {}

for name in models:
  all_results[name] = {'mae': []}
n_splits = 5 # K=5 Folds

# Stratified 5-Fold Cross-Validation, shuffled
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for fold_idx, (train_index, test_index) in enumerate(skf.split(X, y)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  # 1. Split data
  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  for model_name, model in models.items():
    print(f"Training {model_name}...")

    # Train
    model.fit(X_train, y_train)

    # Predict
    y_pred = model.predict(X_test)

    # Calculate metrics
    mae = mean_absolute_error(y_test, y_pred)
    print(mae)
    # Store results
    all_results[model_name]["mae"].append(mae)
    lr_mae.append(mae)

In [ ]:
average_losses = np.mean(lr_mae, axis=0)
print(average_losses)


In [ ]:
baseline_pred = np.full_like(y, y.mean())

# Evaluate the baseline
baseline_mae = mean_absolute_error(y, baseline_pred)
print(f"Baseline MSE (using mean target): {baseline_mae:.4f}")

In [ ]:
# Gather importances from the models (from the last fold)
importances = {}

importances['Random Forest'] = models['Random Forest Regressor'].feature_importances_




fig, axes = plt.subplots(1, 3, figsize=(18, 6))
axes = axes.flatten()
features = X.columns

for i, (model_name, imp) in enumerate(importances.items()):
  # Sort features by importance for a cleaner plot
  sorted_idx = np.argsort(imp)

  ax = axes[i]
  ax.barh(features[sorted_idx], imp[sorted_idx])
  ax.set_title(f"{model_name} Feature Importance")
  ax.set_xlabel("Importance Score")

plt.tight_layout()
plt.show()

In [ ]:
# Task 2: Write your code here:
df_clean['Delivery_Time'].hist(figsize=(12, 8))
plt.tight_layout()
plt.show()

In [ ]:
# Task Bonus: Write your code here: